# 08. 전처리 — 분석용 패널 만들기

- 입력: `data/processed/panel_mid.csv` (07_ai_exposure_eda.ipynb가 만든 직업 중분류 × 반기 패널), `data/processed/eci_ref.csv`
- 출력: `data/processed/panel_analysis.csv`, `data/processed/panel_analysis_codebook.csv`
- 경로는 `src/config.py`를 따른다.

제외 규칙은 설계 문서 8-1절에서 결과를 보기 전에 정한 것이다. 바꾸려면 설계 문서에 이유를 남긴다.

In [ ]:
import sys

sys.path.append("..")  # 노트북 폴더 기준으로 src를 찾는다 (README 6절)
from src import config

import numpy as np
import pandas as pd

try:
    display
except NameError:  # 스크립트로 실행할 때
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)

OUT = config.DATA_PROCESSED
p = pd.read_csv(OUT / "panel_mid.csv", dtype={"code": str, "period": str})
eci = pd.read_csv(OUT / "eci_ref.csv", dtype={"period": str})

EXCL_MAIN = ["11", "12", "13", "14", "15", "62"]   # 청년 0명인 칸이 있는 직업
EXCL_ROBUST = EXCL_MAIN + ["27", "63", "88"]       # + 청년 오차 표시가 시점 절반 이상인 직업
COVID = ["202001", "202002", "202101", "202102"]   # 2020 상 ~ 2021 하

## 1. 시점 변수

`k_post`: 2023년 상반기 = 1 … 2026년 상반기 = 7, 그 전은 0. H1 식에서 "2023년 이후 기울기 변화"를 재는 변수다.

In [ ]:
order = {per: i for i, per in enumerate(sorted(p.period.unique()))}
p["k_post"] = (p.period.map(order) - order["202202"]).clip(lower=0)

## 2. AI 성능(ECI)

2023년 이전은 자료가 없어 빈칸이다. `eci_lag1`은 한 반기 전 값으로, 영향이 늦게 나타나는 경우를 볼 때(H3) 쓴다.

In [ ]:
eci = eci.sort_values("period")
eci["eci_lag1"] = eci.eci.shift(1)
p = p.merge(eci[["period", "eci", "eci_lag1", "d_eci"]], on="period", how="left")

## 3. 결과 변수와 가중치

- `lr_y` = ln(청년) − ln(30~59세): 주 결과변수. 청년이 0명인 칸은 계산할 수 없어 빈칸이다.
- `y_share` = 청년 ÷ 전체(%): 보조 결과변수.
- `w`: 2022년 상·하반기 청년 취업자 평균. 직업마다 하나로 고정한다(결과가 가중치에 섞이지 않도록).

In [ ]:
with np.errstate(divide="ignore"):
    p["lr_y"] = np.log(p.y15_29) - np.log(p.a30_59)
p.loc[p.y15_29 == 0, "lr_y"] = np.nan
p["y_share"] = 100 * p.y15_29 / p.total
p = p.merge(p[p.year == 2022].groupby("code").y15_29.mean().rename("w"), on="code", how="left")

## 4. 표본 표시

In [ ]:
p["in_main"] = ~p.code.isin(EXCL_MAIN)
p["in_robust47"] = ~p.code.isin(EXCL_ROBUST)
p["covid"] = p.period.isin(COVID)

## 5. 검사 — 하나라도 어긋나면 여기서 멈춘다

In [ ]:
main = p[p.in_main]
checks = {
    "(직업, 시점) 조합 중복 없음": not p.duplicated(["code", "period"]).any(),
    "전체 1,512행 (56개 직업 × 27개 시점)": len(p) == 1512 and p.code.nunique() == 56,
    "주 분석 1,350행 (50개 직업)": len(main) == 1350 and main.code.nunique() == 50,
    "검증용 표본 47개 직업": p[p.in_robust47].code.nunique() == 47,
    "주 분석에 lr_y 빈칸·무한대 없음": bool(np.isfinite(main.lr_y).all()),
    "lr_y 빈칸은 청년 0명인 칸만 (88칸)": p.lr_y.isna().sum() == (p.y15_29 == 0).sum() == 88,
    "주 분석 가중치가 모두 양수": bool((main.w > 0).all()),
    "노출도 0~1, 청년 비중 0~100%": bool(p.exp25.between(0, 1).all() and p.y_share.between(0, 100).all()),
    "k_post 0~7이고 post와 일치": bool(p.k_post.between(0, 7).all() and ((p.k_post > 0) == p.post).all()),
    "ECI는 2023년 이후에만 있음": bool(p.loc[p.post, "eci"].notna().all() and p.loc[~p.post, "eci"].isna().all()),
}
for name, ok in checks.items():
    print(("통과  " if ok else "실패  ") + name)
assert all(checks.values()), "검사 실패: 위 목록을 확인"

## 6. 저장

In [ ]:
COLS = ["code", "name", "major", "period", "year", "half", "time", "post", "k_post",
        "total", "y15_29", "a30_39", "a40_49", "a50_59", "a60p", "a30_59", "y_flag",
        "exp25", "exp23", "exp25_nocross", "exp_z", "terc",
        "eci", "eci_lag1", "d_eci", "lr_y", "y_share", "w", "in_main", "in_robust47", "covid"]
p = p.sort_values(["code", "period"])[COLS]
p.to_csv(OUT / "panel_analysis.csv", index=False, encoding="utf-8-sig")

codebook = pd.DataFrame([
    ("code", "직업 중분류 코드", "글자 '11'~'99'", "KOSIS DT_1ES3B28J", "글자로 읽기 (dtype={'code': str})"),
    ("name", "직업 이름", "", "KOSIS DT_1ES3B28J", ""),
    ("major", "직업 대분류 번호", "1~9", "code의 첫 자리", ""),
    ("period", "시점", "'201301' = 2013 상반기, '201302' = 하반기", "KOSIS DT_1ES3B28J", "글자로 읽기 (dtype={'period': str})"),
    ("year", "연도", "", "period", ""),
    ("half", "반기", "1 = 상반기(4월 조사), 2 = 하반기(10월 조사)", "period", ""),
    ("time", "소수로 쓴 연도", "2013.29 = 2013년 4월", "연도 + 조사월/12", "그래프 x축용"),
    ("post", "AI 이후 여부", "True/False", "2023 상반기부터 True", "ChatGPT 출시(2022-11-30)가 2022 하반기 조사(10월)보다 늦음"),
    ("k_post", "AI 이후 몇 번째 반기", "0~7", "2023 상 = 1 … 2026 상 = 7, 그 전 0", "H1 기울기 변화 변수"),
    ("total", "전체 취업자", "천명", "KOSIS DT_1ES3B28J", ""),
    ("y15_29", "청년(15~29세) 취업자", "천명", "KOSIS DT_1ES3B28J", "'-'(해당자 없음)는 0으로 바꿈"),
    ("a30_39", "30대 취업자", "천명", "KOSIS DT_1ES3B28J", ""),
    ("a40_49", "40대 취업자", "천명", "KOSIS DT_1ES3B28J", ""),
    ("a50_59", "50대 취업자", "천명", "KOSIS DT_1ES3B28J", ""),
    ("a60p", "60세 이상 취업자", "천명", "KOSIS DT_1ES3B28J", ""),
    ("a30_59", "30~59세 취업자", "천명", "a30_39 + a40_49 + a50_59", "비교군"),
    ("y_flag", "청년 추정치 오차가 큼", "True/False", "KOSIS의 '*' 표시 (상대표준오차 25% 이상)", "전체 칸의 14.7%"),
    ("exp25", "AI 노출도 (주 분석)", "0~1", "ILO WP140 2025 점수 → 연계표로 한국 세분류 평균 → 중분류 평균", "단순평균 (취업자 가중 아님)"),
    ("exp23", "AI 노출도 (2023 점수)", "0~1", "exp25와 같은 방법, ILO 2023 점수", "검증용"),
    ("exp25_nocross", "AI 노출도 (대분류 벗어난 연결 제외)", "0~1", "exp25와 같은 방법", "검증용. 직업 88은 빈칸"),
    ("exp_z", "AI 노출도 표준화 값", "평균 0, 표준편차 1", "(exp25 − 평균) ÷ 표준편차, 56개 직업 기준", "회귀 결과를 '1표준편차당'으로 읽을 때"),
    ("terc", "노출도 3분위", "하 / 중 / 상", "exp25 순위로 3등분 (19 / 18 / 19개 직업)", "그림용"),
    ("eci", "그 시점의 최고 AI 성능", "ECI 점수", "Epoch eci_scores.csv, 조사 기준일까지 나온 모델 중 최고점", "2023년 이전 빈칸. 2026-11-11 파일 기준"),
    ("eci_lag1", "한 반기 전 ECI", "ECI 점수", "eci를 한 반기 늦춤", "2023 상반기 포함 빈칸"),
    ("d_eci", "직전 반기 대비 ECI 상승폭", "ECI 점수", "eci − eci_lag1", "H3"),
    ("lr_y", "주 결과변수: ln(청년 ÷ 30~59세)", "로그 (0.05 감소 ≈ 상대 비율 5% 감소)", "ln(y15_29) − ln(a30_59)", "청년 0명인 칸은 빈칸 (88칸, 모두 주 분석 제외 직업)"),
    ("y_share", "보조 결과변수: 청년 비중", "%", "100 × y15_29 ÷ total", "60세 이상 증가의 영향이 섞임"),
    ("w", "가중치", "천명", "2022년 상·하반기 청년 취업자 평균, 직업마다 고정", ""),
    ("in_main", "주 분석 표본", "True/False", "직업 11·12·13·14·15·62 제외", "50개 직업 × 27개 시점 = 1,350행"),
    ("in_robust47", "검증용 표본", "True/False", "위 6개 + 27·63·88 제외", "47개 직업"),
    ("covid", "코로나 시기", "True/False", "2020 상반기 ~ 2021 하반기", "제외 분석용"),
], columns=["열", "뜻", "단위·값", "출처·계산", "비고"])
assert list(codebook["열"]) == COLS, "변수 설명표와 실제 열이 다름"
codebook.to_csv(OUT / "panel_analysis_codebook.csv", index=False, encoding="utf-8-sig")
print("저장:", OUT / "panel_analysis.csv", p.shape, "|", OUT / "panel_analysis_codebook.csv", codebook.shape)
display(p[p.code.isin(["33", "45"]) & p.period.isin(["202202", "202601"])])

## 쓰는 법

```python
df = pd.read_csv(config.DATA_PROCESSED / "panel_analysis.csv", dtype={"code": str, "period": str})
main = df[df.in_main]          # 주 분석 (50개 직업)
robust = df[df.in_robust47]    # 검증용 (47개 직업)
no_covid = main[~main.covid]   # 코로나 시기 제외
```